# mình chọn 1 thuật toán (Naive Bayes.....)


# 1. Import thư viện

In [4]:
import pandas as pd
import numpy as np
import re
import string
import nltk
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

1.1 đọc dữ liệu

In [5]:
import pandas as pd

df = pd.read_csv("mail_data.csv")
print(df.head())

  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


1.2 kiểm tra dữ liệu


In [6]:
print(df.isnull().sum())
df = df.fillna("")

Category    0
Message     0
dtype: int64


1.3. kiểm tra thông tin

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Category  5572 non-null   str  
 1   Message   5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


1.4. Kiểm tra kích thước:

In [8]:
df.shape

(5572, 2)

# 2.Tiền xử lý dữ liệu (tuấn)



2.1. Chuyển văn bản về chữ thường 

In [9]:
df['Message'] = df['Message'].str.lower()

2.2 Loại bỏ ký tự đặc biệt và số

In [10]:
import re
import string

def clean_text(text):

    # Xóa ký tự đặc biệt
    text = re.sub(f"[{string.punctuation}]", "", text)

    # Xóa số
    text = re.sub(r"\d+", "", text)

    return text

df["Message"] = df["Message"].apply(clean_text)

2.3 chuẩn hóa dữ liệu

In [11]:
df["Message"] = df["Message"].str.replace(r"\s+", " ", regex=True).str.strip()

2.4 Bỏ stop words

In [12]:
import nltk
nltk.download("stopwords")

from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df["Message"] = df["Message"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


2.2 Mã hóa nhãn


In [13]:
# Chỉ lấy các mẫu ham và spam
df = df[df["Category"].isin(["ham", "spam"])]

# Tách dữ liệu
X = df["Message"]
y = df["Category"]

# Kiểm tra phân bố nhãn
print("Kiểm tra phân bố nhãn:")
print(y.value_counts())

print("\nKiểm tra phân bố nhãn (theo tỉ lệ):")
print(y.value_counts(normalize=True))

# Mã hóa nhãn
y = y.map({
    "ham": 0,
    "spam": 1
})

print("\nNhãn sau khi mã hóa:")
print(y.head())

Kiểm tra phân bố nhãn:
Category
ham     4825
spam     747
Name: count, dtype: int64

Kiểm tra phân bố nhãn (theo tỉ lệ):
Category
ham     0.865937
spam    0.134063
Name: proportion, dtype: float64

Nhãn sau khi mã hóa:
0    0
1    0
2    1
3    0
4    0
Name: Category, dtype: int64


# 3 xử lý dữ liệu


3.1 Chia tập Train/Test

In [14]:
from sklearn.model_selection import train_test_split

# Chia dữ liệu thành Train và Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Kiểm tra kích thước dữ liệu
print("Kích thước tập Train:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nKích thước tập Test:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Kích thước tập Train:
X_train: (4457,)
y_train: (4457,)

Kích thước tập Test:
X_test: (1115,)
y_test: (1115,)


kiểm tra phân bố nhãn sau khi chia 


In [15]:
print("Train:")
print(y_train.value_counts(normalize=True))

print("\nTest:")
print(y_test.value_counts(normalize=True))

Train:
Category
0    0.865829
1    0.134171
Name: proportion, dtype: float64

Test:
Category
0    0.866368
1    0.133632
Name: proportion, dtype: float64


3.2 Chuyển đổi văn bản bằng TF-IDF


Mục đích: Chuyển email từ dạng văn bản sang dạng vector số để mô hình Naive Bayes có thể xử lý.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Khởi tạo TF-IDF
tfidf = TfidfVectorizer()

# Huấn luyện TF-IDF trên tập Train
X_train = tfidf.fit_transform(X_train)

# Chuyển tập Test sang vector
X_test = tfidf.transform(X_test)

kết quả

In [17]:
print("Kích thước sau TF-IDF")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

Kích thước sau TF-IDF
X_train: (4457, 7431)
X_test : (1115, 7431)


xem thêm một số từ khóa


In [18]:
print("Số lượng đặc trưng:", len(tfidf.get_feature_names_out()))

print("\n20 từ đầu tiên:")
print(tfidf.get_feature_names_out()[:20])

Số lượng đặc trưng: 7431

20 từ đầu tiên:
['aa' 'aah' 'aaniye' 'aathilove' 'aathiwhere' 'ab' 'abbey' 'abdomen'
 'abeg' 'abelu' 'aberdeen' 'abi' 'ability' 'abiola' 'abj' 'able'
 'abnormally' 'aboutas' 'absence' 'absolutely']


# 4.Huấn luyện mô hình(Hào)
Train Naive Bayes
Fit model với train data
Dự đoán trên test data

In [19]:
nb_classifier = MultinomialNB()
nb_classifier.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7431)","[[0.33,1.27,0.1 ,...,0.17,1.35,0.33], [0. ,0. ,0. ,...,0. ,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7431)","[[-9.46,-8.92,-9.65,...,-9.59,-8.89,-9.46], [-9.16,-9.16,-9.16,...,-9.16,-9.16,-9.16]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7431


# 5.Đánh giá (hào)
Tính Accuracy(hào)
Precision, Recall, F1-score(hào)
Confusion Matrix(hào)


In [20]:
y_pred = nb_classifier.predict(X_test)

accuracy = accuracy_score(y_test, y_pred) * 100
print(f'Accurary: {accuracy}')

Accurary: 96.32286995515696


In [21]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



# 6.demo dự đoán email mới(hoàng)
Nhập một email mới (không có trong dataset).
Tiền xử lý email giống các bước trước.
Chuyển email thành vector bằng TF-IDF.
Đưa vào mô hình Naive Bayes để dự đoán.
Hiển thị kết quả:
Spam
Ham (Not Spam)

Ví dụ:

Email 1 → Kết quả: Spam
Email 2 → Kết quả: Ham

# 7.Kết luận(hoàng)
Công việc cần làm:
Trình bày kết quả đánh giá của mô hình:
Accuracy
Precision
Recall
F1-score
Nhận xét mô hình:
Độ chính xác đạt bao nhiêu %.
Mô hình có phân loại tốt hay không.
Nêu ưu điểm:
Huấn luyện nhanh.
Dự đoán nhanh.
Phù hợp với bài toán phân loại văn bản (Spam Email).
Nêu hạn chế:
Giả định các đặc trưng độc lập.
Có thể dự đoán sai với email có nội dung phức tạp hoặc chưa từng xuất hiện trong dữ liệu huấn luyện.
Kết luận chung:
Naive Bayes phù hợp cho bài toán Spam Email Classification và đạt kết quả tốt trên dataset của nhóm.